In [1]:
import json
import os
import csv
from datetime import datetime

def build_full_borderlands4_dictionary():
    # ========== НАСТРОЙКА ПУТЕЙ ==========
    INPUT_DIR = "../data/dictionary/raw_files"   # откуда берём сырые JSON
    OUTPUT_DIR = "../data/dictionary/result"     # куда кладём результат
    # ======================================

    # Создаём выходную папку, если её нет
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Формируем имена выходных файлов
    timestamp = datetime.now().strftime("%Y_%m_%d_%H%M")  # добавили часы и минуты
    output_json = "dictionary_en_ru.json"                # всегда одинаковое имя
    output_csv = f"borderlands_4_dictionary_en_ru_{timestamp}.csv"

    # Полные пути
    json_path = os.path.join(OUTPUT_DIR, output_json)
    csv_path = os.path.join(OUTPUT_DIR, output_csv)

    print(f"=== СБОРКА ПОЛНОГО СЛОВАРЯ BORDERLANDS 4 [{timestamp}] ===")
    print(f"Входная папка: {INPUT_DIR}")
    print(f"Выходная папка: {OUTPUT_DIR}\n")

    master_en = {}  # hash -> info
    master_ru = {}  # hash -> info

    # Автоматически находим все пары _en и _ru файлов в INPUT_DIR
    all_files = os.listdir(INPUT_DIR)
    en_files = [f for f in all_files if f.endswith('_en.json')]
    ru_files = [f for f in all_files if f.endswith('_ru.json')]

    print(f"Найдено EN файлов: {len(en_files)}")
    print(f"Найдено RU файлов: {len(ru_files)}\n")

    # 2. Считываем все EN файлы (сохраняя источник DLC/Базы)
    for f_name in en_files:
        source_name = f_name.replace('_en.json', '')
        file_path = os.path.join(INPUT_DIR, f_name)
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                count = 0
                for ns, items in data.items():
                    if isinstance(items, dict):
                        for hash_key, text in items.items():
                            master_en[hash_key] = {
                                "namespace": ns,
                                "text": text,
                                "source": source_name
                            }
                            count += 1
                print(f"[+] Загружен EN: {f_name:<18} | Строк: {count}")
        except Exception as e:
            print(f"[!] Ошибка чтения {file_path}: {e}")

    # 3. Считываем все RU файлы
    for f_name in ru_files:
        source_name = f_name.replace('_ru.json', '')
        file_path = os.path.join(INPUT_DIR, f_name)
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                count = 0
                for ns, items in data.items():
                    if isinstance(items, dict):
                        for hash_key, text in items.items():
                            master_ru[hash_key] = {
                                "namespace": ns,
                                "text": text,
                                "source": source_name
                            }
                            count += 1
                print(f"[+] Загружен RU: {f_name:<18} | Строк: {count}")
        except Exception as e:
            print(f"[!] Ошибка чтения {file_path}: {e}")

    # 4. Объединяем ВСЕ хэши без исключений
    all_hashes = set(master_en.keys()) | set(master_ru.keys())
    print(f"\nВсего уникальных хэшей со всех DLC и базовой игры: {len(all_hashes)}")

    combined_dict = {}
    csv_rows = []

    for h in all_hashes:
        en_info = master_en.get(h, {})
        ru_info = master_ru.get(h, {})

        en_text = en_info.get("text", "")
        ru_text = ru_info.get("text", "")
        namespace = en_info.get("namespace") or ru_info.get("namespace", "")
        source = en_info.get("source") or ru_info.get("source", "")

        combined_dict[h] = {
            "source": source,
            "namespace": namespace,
            "en": en_text,
            "ru": ru_text
        }

        csv_rows.append({
            "hash": h,
            "source": source,
            "namespace": namespace,
            "en": en_text,
            "ru": ru_text
        })

    # Сортируем строки по алфавиту источника (Game -> Banjo -> Cello и т.д.)
    csv_rows.sort(key=lambda x: (x["source"], x["hash"]))

    # 5. Записываем в JSON
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(combined_dict, f, ensure_ascii=False, indent=4)
    print(f"\nУспешно создан JSON: {json_path}")

    # 6. Записываем в CSV (с поддержкой кириллицы в Excel)
    with open(csv_path, 'w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=["hash", "source", "namespace", "en", "ru"])
        writer.writeheader()
        writer.writerows(csv_rows)
    print(f"Успешно создан CSV:  {csv_path}")

    print("\nСБОРКА ЗАВЕРШЕНА! Ни один файл и ни один хэш не упущен.")

if __name__ == "__main__":
    build_full_borderlands4_dictionary()

=== СБОРКА ПОЛНОГО СЛОВАРЯ BORDERLANDS 4 [2026_08_06_2008] ===
Входная папка: ../data/dictionary/raw_files
Выходная папка: ../data/dictionary/result

Найдено EN файлов: 9
Найдено RU файлов: 9

[+] Загружен EN: Banjo_en.json      | Строк: 750
[+] Загружен EN: Cello_en.json      | Строк: 1217
[+] Загружен EN: Cowbell_en.json    | Строк: 34165
[+] Загружен EN: Game_en.json       | Строк: 113349
[+] Загружен EN: Harp_en.json       | Строк: 1920
[+] Загружен EN: Mandolin_en.json   | Строк: 1389
[+] Загружен EN: Raid1_en.json      | Строк: 187
[+] Загружен EN: Raid2_en.json      | Строк: 635
[+] Загружен EN: Tuba_en.json       | Строк: 793
[+] Загружен RU: Banjo_ru.json      | Строк: 723
[+] Загружен RU: Cello_ru.json      | Строк: 1120
[+] Загружен RU: Cowbell_ru.json    | Строк: 33085
[+] Загружен RU: Game_ru.json       | Строк: 110202
[+] Загружен RU: Harp_ru.json       | Строк: 1761
[+] Загружен RU: Mandolin_ru.json   | Строк: 1045
[+] Загружен RU: Raid1_ru.json      | Строк: 140
[+] Заг